In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r".\dataset")

excel_paths = sorted(DATA_DIR.glob('*.xls'))

print("Cek struktur kolom dataset ISPU")

for p in excel_paths:
    try:
        dfs = pd.read_html(p)
        
        df = dfs[0]
        kolom = df.columns.tolist()
        
        print(f"\nFile: {p.name}")
        print(f"Jumlah kolom: {len(kolom)}")
        print(f"Daftar kolom: {kolom}")
    except ValueError as ve:
        print(f"Gagal membaca {p.name}: {ve}")
    except Exception as e:
        print(f"Gagal membaca {p.name}: {e}")

print("\nSelesai cek struktur kolom.")

In [ ]:

# 1. Kamus (Dictionary) untuk pemetaan nama kolom agar seragam
# Format: {'nama_kolom_lama': 'nama_kolom_standar'}
kolom_mapping = {
    'wilayah': 'stasiun',
    'lokasi_spku': 'stasiun',
    'pm_sepuluh': 'pm10',
    'pm_10': 'pm10',
    'pm_duakomalima': 'pm25',
    'sulfur_dioksida': 'so2',
    'karbon_monoksida': 'co',
    'ozon': 'o3',
    'nitrogen_dioksida': 'no2',
    'parameter_pencemar_kritis': 'critical',
    'categori': 'kategori'
}

semua_data = []

print("Mulai gabung data...")

for p in excel_paths:
    try:
        df = pd.read_html(p)[0]
        
        # Bersihkan nama kolom bawaan dulu (jadikan huruf kecil & hapus spasi tersembunyi)
        df.columns = df.columns.str.lower().str.strip()
        
        # 2. Ganti nama kolom menggunakan kamus yang sudah dibuat
        df.rename(columns=kolom_mapping, inplace=True)
        
        semua_data.append(df)
        print(f"Proses file: {p.name}, baris: {len(df)}")
        
    except Exception as e:
        print(f"Gagal proses {p.name}: {e}")

# 3. Gabungkan semua dataframe di dalam list menjadi satu DataFrame utuh
df_master = pd.concat(semua_data, ignore_index=True)

print("\nGabung data selesai.")
print(f"Total baris: {df_master.shape[0]}")
print(f"Total kolom: {df_master.shape[1]}")
print(f"Kolom data: {df_master.columns.tolist()}")

In [ ]:
print("Sanity check data")

print("\n1) Info tipe data dan missing values:")
# info() akan otomatis mencetak ke terminal
df_master.info()

print("\n2) Distribusi kategori:")
print(df_master['kategori'].value_counts(dropna=False))

print("\n3) Distribusi stasiun:")
print(df_master['stasiun'].value_counts(dropna=False))

print("\n4) Contoh nilai unik kolom polutan:")
# Kalau isinya cuma angka, tipe datanya pasti int64 atau float64.
# Kalau tipe datanya 'object', berarti ada teks nyelip. Kita intip 15 nilai unik pertamanya:
kolom_polutan = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
for col in kolom_polutan:
    if col in df_master.columns:
        print(f"- {col}: {df_master[col].unique()[:15]}")

In [ ]:
import numpy as np

print("Data cleaning")

# 1. Bersihkan Kolom Polutan (Ubah '-' jadi NaN, lalu ubah tipe data ke Numerik/Float)
kolom_numerik = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max']
for col in kolom_numerik:
    if col in df_master.columns:
        # pd.to_numeric dengan errors='coerce' akan memaksa text aneh ('-', '---') menjadi NaN
        df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 2. Bersihkan Kolom 'kategori' (Perbaiki typo dan buang data yang bergeser)
if 'kategori' in df_master.columns:
    # Jadikan huruf besar semua dan hilangkan spasi di awal/akhir kata
    df_master['kategori'] = df_master['kategori'].astype(str).str.upper().str.strip()
    
    # Perbaiki Typo
    df_master['kategori'] = df_master['kategori'].replace({
        'TIDAKSEHAT': 'TIDAK SEHAT',
        'TIDAKADADATA': 'TIDAK ADA DATA'
    })
    
    # Kita hanya ambil baris data yang kategorinya valid secara aturan ISPU
    kategori_valid = ['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT', 'BERBAHAYA']
    df_master = df_master[df_master['kategori'].isin(kategori_valid)]

# 3. Bersihkan Kolom 'stasiun' (Seragamkan nama)
def standarisasi_stasiun(nama):
    nama = str(nama).upper()
    if 'DKI1' in nama: return 'DKI1 (BUNDERAN HI)'
    elif 'DKI2' in nama: return 'DKI2 (KELAPA GADING)'
    elif 'DKI3' in nama: return 'DKI3 (JAGAKARSA)'
    elif 'DKI4' in nama: return 'DKI4 (LUBANG BUAYA)'
    elif 'DKI5' in nama: return 'DKI5 (KEBON JERUK)'
    else: return np.nan # Kalau ada nama aneh (kayak 'SEDANG'), jadikan NaN

if 'stasiun' in df_master.columns:
    df_master['stasiun'] = df_master['stasiun'].apply(standarisasi_stasiun)
    # Buang baris yang stasiunnya NaN (imbas dari data bergeser tadi)
    df_master.dropna(subset=['stasiun'], inplace=True)

# Cek hasil akhirnya
print("\nInfo data setelah cleaning:")
df_master.info()

print("\nDistribusi kategori:")
print(df_master['kategori'].value_counts())

print("\nDistribusi stasiun:")
print(df_master['stasiun'].value_counts())

In [ ]:
print("Handling missing values dan outlier")

# 1. Hapus kolom 'pm25' (terlalu banyak NaN) dan 'periode_data' (tidak relevan untuk model)
kolom_dihapus = ['pm25', 'periode_data']
df_master.drop(columns=[col for col in kolom_dihapus if col in df_master.columns], inplace=True)

# 2. Hapus 1 baris kelas 'BERBAHAYA'
df_master = df_master[df_master['kategori'] != 'BERBAHAYA']

# 3. Imputasi (Isi data kosong) pada polutan lain dengan nilai MEDIAN berdasarkan 'stasiun'
kolom_imputasi = ['pm10', 'so2', 'co', 'o3', 'no2', 'max']
for col in kolom_imputasi:
    # Mengisi NaN dengan median dari stasiun yang sama
    df_master[col] = df_master.groupby('stasiun')[col].transform(lambda x: x.fillna(x.median()))

# 4. Drop sisa baris yang mungkin masih NaN di kolom target (kategori) atau stasiun
df_master.dropna(subset=['kategori', 'stasiun'], inplace=True)

print("Missing values setelah imputasi:")
print(df_master.isna().sum())

print(f"Bentuk data akhir: {df_master.shape}")

In [ ]:
print("Feature engineering")

# 1. Membuang Data Leakage & Kolom Non-Prediktif
# 'tanggal' juga kita buang dulu karena untuk tahap awal ini kita fokus ke angka polutannya
kolom_buang = ['tanggal', 'max', 'critical']
df_model = df_master.drop(columns=[col for col in kolom_buang if col in df_master.columns]).copy()

# 2. Encoding Kolom Target (Kategori)
# Karena tingkat polusi itu berurutan (Ordinal), kita ubah ke angka urut
map_kategori = {
    'BAIK': 0,
    'SEDANG': 1,
    'TIDAK SEHAT': 2,
    'SANGAT TIDAK SEHAT': 3
}
df_model['kategori'] = df_model['kategori'].map(map_kategori)

# 3. Encoding Kolom Stasiun (One-Hot Encoding)
# Karena stasiun tidak ada tingkatannya, kita jadikan kolom boolean (0/1) menggunakan get_dummies
df_model = pd.get_dummies(df_model, columns=['stasiun'], drop_first=True)

# Memastikan semua data bertipe boolean (True/False) dari get_dummies diubah ke int (1/0)
for col in df_model.select_dtypes(include=['bool']).columns:
    df_model[col] = df_model[col].astype(int)

print("Contoh data siap model:")
print(df_model.head())

print(f"Bentuk data akhir: {df_model.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

print("Modeling dan evaluasi")

# 1. Pisahkan Fitur (X) dan Target/Jawaban (y)
X = df_model.drop(columns=['kategori'])
y = df_model['kategori']

# 2. Bagi data (Data Splitting)
# Kita pakai 80% data untuk belajar (Training), 20% untuk ujian (Testing)
# stratify=y memastikan proporsi kelas 'BAIK', 'SEDANG', dsb seimbang di tes & latih
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Data training: {X_train.shape[0]} baris")
print(f"Data testing: {X_test.shape[0]} baris")

# 3. Inisialisasi Model Random Forest
# class_weight='balanced' ini senjata rahasia kita karena data 'SANGAT TIDAK SEHAT' jumlahnya sedikit
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# 4. Latih Model (Fit)
print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training selesai.")

# 5. Uji Model dengan Data Testing (Predict)
y_pred = rf_model.predict(X_test)

# 6. Lihat Nilai Rapornya
print("Hasil evaluasi model")
print(f"Akurasi: {accuracy_score(y_test, y_pred) * 100:.2f}%")

print("Classification report:")
target_names = ['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT']
print(classification_report(y_test, y_pred, target_names=target_names))

# Bonus: Cek fitur (polutan) apa yang paling ngaruh ke target
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
print(feature_importances)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
import joblib

print("Visualisasi confusion matrix")

# 1. Membuat Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))

# Menggunakan Seaborn Heatmap untuk pewarnaan
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            annot_kws={"size": 12}) # Perbesar angka di dalam kotak

plt.title('Confusion Matrix - Prediksi Kualitas Udara (ISPU)', fontsize=14, pad=15)
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Jawaban Asli (Aktual)', fontsize=12)
plt.tight_layout()

# Simpan gambarnya ke dalam folder yang sama dengan skrip ini
nama_file_gambar = 'confusion_matrix.png'
plt.savefig(nama_file_gambar, dpi=300)
print(f"Confusion matrix disimpan: '{nama_file_gambar}'")

print("\nCross validation (5-fold)")

# 2. Cross Validation
# Kita melipat data 5 kali (cv=5) untuk menguji kestabilan model di berbagai skenario pemotongan data
print("Proses cross validation...")
scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')

print(f"Skor per fold: {np.round(scores * 100, 2)}")
print(f"Rata-rata akurasi: {scores.mean() * 100:.2f}%")
print(f"Standar deviasi: {scores.std():.4f}")

print("\nSimpan model")

# 3. Export Model ke file .pkl
nama_file_model = 'model_ispu_randomforest.pkl'
joblib.dump(rf_model, nama_file_model)
print(f"Model disimpan: '{nama_file_model}'")
print("Model siap digunakan.")

In [ ]:

DATA_DIR = Path(r".\dataset")

# Ekstrak path untuk kedua file 2018 tersebut
file_1 = DATA_DIR / "Data_Indeks Standar Pencemaran Udara di SPKU DKI Jakarta Tahun 2018 - tabel.xls"
file_2 = DATA_DIR / "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2018 - tabel.xls"

def investigasi_file(path_file, label):
    print(f"\n{'='*50}")
    print(f"🔍 INVESTIGASI: {label}")
    print(f"Nama File: {path_file.name}")
    print(f"{'='*50}")
    
    try:
        # Menggunakan read_html karena filenya aslinya berformat HTML
        df = pd.read_html(path_file)[0]
        
        # Bersihkan nama kolom agar mudah dicari
        df.columns = df.columns.str.lower().str.strip()
        
        # Cek nama kolom untuk lokasi/stasiun
        kolom_stasiun = 'stasiun' if 'stasiun' in df.columns else 'lokasi_spku'
        
        print(f"Total Baris Data : {len(df)}")
        
        if kolom_stasiun in df.columns:
            print(f"Jumlah Stasiun   : {df[kolom_stasiun].nunique()}")
            print(f"\nRincian Jumlah Data per Stasiun:")
            print(df[kolom_stasiun].value_counts().to_string())
        else:
            print("Tidak menemukan kolom stasiun/lokasi!")
            
        print("\nContoh 3 Baris Pertama:")
        print(df.head(3).to_string())
        
    except Exception as e:
        print(f"Gagal membaca file. Error: {e}")

# Eksekusi pengecekan
if file_1.exists() and file_2.exists():
    investigasi_file(file_1, "FILE 2018 VERSI 1 (SPKU)")
    investigasi_file(file_2, "FILE 2018 VERSI 2 (ISPU)")
else:
    print("Pastikan kedua nama file 2018 di atas sudah benar dan ada di dalam folder dataset.")

In [ ]:

excel_paths = sorted(DATA_DIR.glob('*.xls'))

print("=== AUDIT TOTAL BARIS & DISTRIBUSI LOKASI DATASET ===\n")
print(f"Total file yang diperiksa: {len(excel_paths)} file.\n")

for p in excel_paths:
    try:
        # Membaca tabel HTML di dalam file .xls
        df = pd.read_html(p)[0]
        df.columns = df.columns.str.lower().str.strip()
        
        # Mencari kolom yang berisi informasi stasiun/lokasi
        kolom_stasiun = None
        for col in ['stasiun', 'lokasi_spku', 'wilayah']:
            if col in df.columns:
                kolom_stasiun = col
                break
                
        total_baris = len(df)
        
        print(f"File: {p.name}")
        print(f"  -> Total Baris : {total_baris}")
        
        if kolom_stasiun:
            # Mengambil daftar lokasi unik yang ada di dalam file tersebut
            stasiun_unik = df[kolom_stasiun].dropna().unique().tolist()
            print(f"  -> Nama Kolom  : '{kolom_stasiun}'")
            print(f"  -> Total Lokasi: {len(stasiun_unik)} stasiun unik")
            print(f"  -> Daftar / Sampel Lokasi: {stasiun_unik[:5]}")
        else:
            print("  -> ⚠️ Peringatan: Kolom stasiun/lokasi tidak ditemukan!")
        print("-" * 60)
        
    except Exception as e:
        print(f"❌ Gagal memeriksa {p.name}. Error: {e}")
        print("-" * 60)

print("\nAudit selesai.")

In [ ]:

# Daftar 7 file yang cuma punya 300an baris
file_300an = [
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2011 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2012 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2013 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2015 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2019 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2020 - tabel.xls",
    "Indeks_Standar Pencemaran Udara (ISPU) Tahun 2022 - tabel.xls"
]

print("=== INVESTIGASI MENDALAM DATA 300-an BARIS ===\n")

for nama_file in file_300an:
    p = DATA_DIR / nama_file
    if p.exists():
        try:
            df = pd.read_html(p)[0]
            df.columns = df.columns.str.lower().str.strip()
            
            # Cari kolom stasiun
            kolom_stasiun = None
            for col in ['stasiun', 'lokasi_spku', 'wilayah']:
                if col in df.columns:
                    kolom_stasiun = col
                    break
            
            print(f"File: {nama_file}")
            print(f"Total Baris Data    : {len(df)}")
            print(f"Total Tanggal Unik  : {df['tanggal'].nunique()}")
            
            if kolom_stasiun:
                print("Distribusi Stasiun  :")
                print(df[kolom_stasiun].value_counts().to_string())
            else:
                print("⚠️ Tidak ada kolom stasiun.")
            print("-" * 50)
            
        except Exception as e:
            print(f"❌ Gagal memeriksa {nama_file}. Error: {e}")
            print("-" * 50)
    else:
        print(f"File tidak ditemukan di folder: {nama_file}")

In [ ]:
FILE_PATH = DATA_DIR / "Data_Indeks Standar Pencemar Udara (ISPU) di Provinsi DKI Jakarta - tabel.xls"

print("=== INVESTIGASI DATASET BARU (3000+ BARIS) ===\n")

if FILE_PATH.exists():
    try:
        # Membaca tabel HTML di dalam file .xls baru
        df = pd.read_html(FILE_PATH)[0]
        df.columns = df.columns.str.lower().str.strip()
        
        print(f"Nama File           : {FILE_PATH.name}")
        print(f"Total Baris Data    : {len(df)}")
        print(f"Daftar Kolom        : {df.columns.tolist()}")
        
        # Cek kolom tanggal jika ada
        if 'tanggal' in df.columns:
            print(f"Total Tanggal Unik  : {df['tanggal'].nunique()}")
            print(f"Rentang Tanggal     : {df['tanggal'].min()} s/d {df['tanggal'].max()}")
        
        # Cari kolom yang berisi informasi stasiun/lokasi
        kolom_stasiun = None
        for col in ['stasiun', 'lokasi_spku', 'wilayah']:
            if col in df.columns:
                kolom_stasiun = col
                break
        
        if kolom_stasiun:
            print(f"\nDistribusi Data Per Stasiun (Kolom '{kolom_stasiun}'):")
            print(df[kolom_stasiun].value_counts().to_string())
        else:
            print("\n⚠️ Kolom stasiun/lokasi tidak ditemukan!")
            
        print("\nContoh 5 Baris Pertama Data:")
        print(df.head().to_string())
        
    except Exception as e:
        print(f"❌ Gagal membaca file. Error: {e}")
else:
    print(f"❌ File tidak ditemukan di path: {FILE_PATH}")
    print("Pastikan nama filenya sudah sesuai dengan yang ada di folder dataset kamu.")

In [ ]:

FILE_PATH = DATA_DIR / "Data_Indeks Standar Pencemar Udara (ISPU) di Provinsi DKI Jakarta - tabel.xls"

print("=== BEDAH DETAIL DATASET (TANPA ASUMSI) ===\n")

if FILE_PATH.exists():
    df = pd.read_html(FILE_PATH)[0]
    df.columns = df.columns.str.lower().str.strip()
    
    print("[1] FAKTA PERIODE WAKTU (KOLOM 'periode_data')")
    periode_unik = sorted(df['periode_data'].dropna().unique().tolist())
    print(f"Total bulan unik yang tercatat : {len(periode_unik)}")
    print(f"Daftar bulan (YYYYMM)          : {periode_unik}\n")
    
    print("[2] FAKTA DUPLIKASI DATA")
    jumlah_duplikat = df.duplicated().sum()
    print(f"Ditemukan baris duplikat (identik 100%) : {jumlah_duplikat} baris\n")
    
    print("[3] FAKTA DISTRIBUSI STASIUN (SETELAH DISATUKAN)")
    def bersihkan_nama_stasiun(nama):
        nama = str(nama).upper()
        if 'DKI1' in nama: return 'DKI1'
        elif 'DKI2' in nama: return 'DKI2'
        elif 'DKI3' in nama: return 'DKI3'
        elif 'DKI4' in nama: return 'DKI4'
        elif 'DKI5' in nama: return 'DKI5'
        else: return nama
        
    df['stasiun_bersih'] = df['stasiun'].apply(bersihkan_nama_stasiun)
    print(df['stasiun_bersih'].value_counts().to_string())
    print()
    
    print("[4] FAKTA DATA KOSONG (MISSING VALUES)")
    print(df.isna().sum().to_string())
    
else:
    print("❌ File tidak ditemukan. Cek kembali path/nama filenya.")